In [16]:
using JuMP
using Gurobi
using CSV
using DataFrames
using NPZ
using LinearAlgebra

## Data Ingestion and Index Sets

### Inputs
- `distance.csv`: arena-to-arena travel distance matrix (`30 x 30`).
- `W.npy`: objective weight tensor `w[i,j,d]` for away team `i`, home team `j`, day `d`.
- `T_correct.npy`: prior-season schedule tensor used for warm start and baseline objective.
- `G.csv`: required total games per unordered team pair.

### Modeling Indexing
- Teams are indexed as `1:30`.
- Season days are indexed as `1:174`.
- Weekly partitions are precomputed for enforcing rolling weekly constraints.

### Reproducibility Notes
- Keep all input files in the repository root (same directory as this notebook).
- Use relative paths only to ensure the notebook runs on any machine.

In [17]:
Teams = 1:30
Days = 1:174

dist_df = CSV.read("distance.csv", DataFrame)
dist = Matrix(dist_df)
w = npzread("W.npy")
T = npzread("T_correct.npy")
G = Matrix(CSV.read("G.csv", DataFrame))[:, 2:end]

no_game_days = [15, 38, 57, 64, 116, 117, 118, 119, 120, 173]
christmas_day = 65
opening_day = 1

Weeks = [
    1:7, 8:14, 15:21, 22:28, 29:35, 36:42, 43:49,
    50:56, 57:63, 64:70, 71:77, 78:84, 85:91,
    92:98, 99:105, 106:112, 113:119, 120:126,
    127:133, 134:140, 141:147, 148:154, 155:161,
    162:168, 169:174
]

25-element Vector{UnitRange{Int64}}:
 1:7
 8:14
 15:21
 22:28
 29:35
 36:42
 43:49
 50:56
 57:63
 64:70
 71:77
 78:84
 85:91
 92:98
 99:105
 106:112
 113:119
 120:126
 127:133
 134:140
 141:147
 148:154
 155:161
 162:168
 169:174

## Baseline Objective Evaluation

This section evaluates the prior-season schedule tensor `T` under the current objective coefficients `w`. The resulting scalar provides a baseline for comparison against the optimized schedule.

In [18]:
function compute_objective(T, w, Teams, Days)
    return sum(w[i, j, d] * T[i, j, d] for i in Teams, j in Teams, d in Days)
end

last_year_obj = compute_objective(T, w, Teams, Days)
println("Baseline objective value = ", last_year_obj)

Last year's objective value = 562.7152742


## MILP Formulation and Solve

This section builds and solves a binary mixed-integer linear program with JuMP + Gurobi.

### Decision Variable
- `x[i,j,d] = 1` iff away team `i` plays at home team `j` on day `d`.

### Objective
- Maximize total weighted schedule quality: `sum(w[i,j,d] * x[i,j,d])`.

### Constraint Families
- Daily participation limit (at most one game per team-day).
- Season game total per team.
- Pairwise matchup totals from matrix `G`.
- Special-day and no-game-day rules.
- Weekly game-count bounds (`0 <= weekly games <= 5`).
- Daily league-wide game-count bounds (`4 <= daily games <= 10` on normal days).
- Global no-3-in-3 restriction.
- Back-to-back linking and weekly cap (`<= 1` B2B per team per week).
- Travel-distance feasibility cuts for consecutive-day transitions (`>= 1600` miles disallowed).

### Solve Strategy
- Warm start from prior schedule tensor `T`.
- Use multi-threaded Gurobi solve and export incumbent schedule.

In [ ]:
env = Gurobi.Env()
model = Model(() -> Gurobi.Optimizer(env))

set_optimizer_attribute(model, "Threads", 8) 


@variable(model, x[i in Teams, j in Teams, d in Days], Bin)

@constraint(model, [i in Teams, d in Days], x[i, i, d] == 0)

@objective(model, Max,
    sum(w[i, j, d] * x[i, j, d] for i in Teams, j in Teams, d in Days)
)
@constraint(model, [t in Teams, d in Days],
    sum(x[t, j, d] for j in Teams if j != t) +
    sum(x[i, t, d] for i in Teams if i != t)
    <= 1
)

@constraint(model, [t in Teams],
    sum(x[t, j, d] for j in Teams if j != t for d in Days) +
    sum(x[i, t, d] for i in Teams if i != t for d in Days)
    == 82
)

@constraint(model, [i in Teams, j in Teams; i < j],
    sum(x[i, j, d] + x[j, i, d] for d in Days) == G[i, j]
)
@constraint(model, [d in no_game_days],
    sum(x[i,j,d] for i in Teams, j in Teams if i!=j) == 0
)

@constraint(model,
    sum(x[i, j, christmas_day] for i in Teams, j in Teams if i != j) <= 5
)

@constraint(model,
    sum(x[i, j, opening_day] for i in Teams, j in Teams if i != j) <= 2
)

for days_w in Weeks
    for t in Teams
        @constraint(model,
            sum(x[t, j, d] for d in days_w, j in Teams if j != t) +
            sum(x[i, t, d] for d in days_w, i in Teams if i != t)
            >= 0
        )
        @constraint(model,
            sum(x[t, j, d] for d in days_w, j in Teams if j != t) +
            sum(x[i, t, d] for d in days_w, i in Teams if i != t)
            <= 5
        )
    end
end

special_days = vcat(no_game_days, [christmas_day, opening_day])
normal_days = [d for d in Days if !(d in special_days)]
@constraint(model, [d in normal_days],
    sum(x[i,j,d] for i in Teams, j in Teams if i != j) >= 4
)

@constraint(model, [d in normal_days],
    sum(x[i,j,d] for i in Teams, j in Teams if i != j) <= 10
)

last_day2 = maximum(Days) - 2

for t in Teams, d in 1:last_day2
    @constraint(model,
        (sum(x[t, j, d] for j in Teams if j != t) +
         sum(x[i, t, d] for i in Teams if i != t)) +
        (sum(x[t, j, d + 1] for j in Teams if j != t) +
         sum(x[i, t, d + 1] for i in Teams if i != t)) +
        (sum(x[t, j, d + 2] for j in Teams if j != t) +
         sum(x[i, t, d + 2] for i in Teams if i != t))
        <= 2
    )
end

@variable(model, g[t in Teams, d in Days], Bin)

@constraint(model, [t in Teams, d in Days],
    g[t, d] ==
        sum(x[t, j, d] for j in Teams if j != t) +
        sum(x[i, t, d] for i in Teams if i != t)
)

last_day = maximum(Days) - 1
@variable(model, b2b[t in Teams, d in 1:last_day], Bin)

@constraint(model, [t in Teams, d in 1:last_day],
    b2b[t,d] >= g[t,d] + g[t,d+1] - 1
)
@constraint(model, [t in Teams, d in 1:last_day],
    b2b[t,d] <= g[t,d]
)
@constraint(model, [t in Teams, d in 1:last_day],
    b2b[t,d] <= g[t,d+1]
)
for days_w in Weeks
    for t in Teams
        @constraint(model,
            sum(b2b[t, d] for d in days_w if d <= last_day) <= 1
        )
    end
end

max_miles = 5000
last_day = maximum(Days) - 1

for i in Teams, a in Teams, b in Teams, d in 1:last_day
    if dist[a, b] >= max_miles
        @constraint(model, x[i, a, d] + x[i, b, d + 1] <= 1)
    end
end

for i in Teams, a in Teams, d in 1:last_day
    if dist[a, i] >= max_miles
        @constraint(model,
            x[i, a, d] + sum(x[k, i, d + 1] for k in Teams if k != i) <= 1
        )
    end
end

for i in Teams, b in Teams, d in 1:last_day
    if dist[i, b] >= max_miles
        @constraint(model,
            sum(x[k, i, d] for k in Teams if k != i) + x[i, b, d + 1] <= 1
        )
    end
end

for i in Teams, j in Teams, d in Days
    set_start_value(x[i, j, d], T[i, j, d])
end

optimize!(model)

Set parameter Username
Set parameter LicenseID to value 2696992
Academic license - for non-commercial use only - expires 2026-08-20
Set parameter Threads to value 8
Set parameter Threads to value 8
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
Threads  8

Optimize a model with 39441 rows, 167010 columns and 2907300 nonzeros
Model fingerprint: 0xa15dd7ac
Variable types: 0 continuous, 167010 integer (167010 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 6e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+01]

User MIP start did not produce a new incumbent solution
User MIP start violates constraint R11676 by 1.000000000

Presolve removed 18160 rows and 86220 columns
Presolve time: 1.07s
Presolved: 21281 rows, 80790 columns, 401658 nonzeros
Variable types: 0 continuou

## Export Incumbent Schedule

Post-solve, this section extracts binary variable assignments `x[i,j,d] = 1` into tabular schedule form:
- `Day`
- `HomeTeam`
- `AwayTeam`
- `Weight`

The exported CSV is used by downstream validation and back-to-back diagnostics.

In [20]:
using JuMP
using DataFrames
using CSV
using MathOptInterface
const MOI = MathOptInterface

term = termination_status(model)
if term != MOI.OPTIMAL && term != MOI.LOCALLY_SOLVED
    @warn "Model not solved to optimality; termination_status = $term"
end

games = DataFrame(
    Day = Int[],
    HomeTeam = Int[],
    AwayTeam = Int[],
    Weight = Float64[],
)

for d in Days, i in Teams, j in Teams
    if i != j && value(x[i, j, d]) > 0.5
        push!(games, (
            Day = d,
            HomeTeam = j,
            AwayTeam = i,
            Weight = w[i, j, d],
        ))
    end
end

CSV.write("schedule_final_5000.csv", games)
println("Exported $(nrow(games)) games to schedule_final_5000.csv")

┌ Warning: Model not solved to optimality; termination_status = INTERRUPTED
└ @ Main In[20]:9


Exported 1230 games to schedule.csv
Exported to schedule.csv


┌ Warning: Model not solved to optimality; termination_status = INTERRUPTED
└ @ Main In[20]:48


## B2B Diagnostics

This section computes per-team and aggregate back-to-back counts from the exported schedule. It also computes travel-distance statistics across consecutive-game days to quantify back-to-back travel burden.

In [21]:
using CSV, DataFrames

sched = CSV.read("schedule_final_5000.csv", DataFrame)
Teams = sort(unique(vcat(sched.HomeTeam, sched.AwayTeam)))

function games_by_day(sched, t)
    df = sched[(sched.HomeTeam .== t) .| (sched.AwayTeam .== t), :]
    return combine(groupby(df, :Day), nrow => :Games)
end

total_b2b = 0
b2b_per_team = Dict{Int, Int}()

for t in Teams
    g = games_by_day(sched, t)
    sort!(g, :Day)

    if any(g.Games .> 1)
        println("WARNING: team $t has multiple games on same day:")
        println(g[g.Games .> 1, :])
    end

    count_t = 0
    for i = 1:(size(g, 1) - 1)
        if g.Day[i + 1] == g.Day[i] + 1 && g.Games[i] == 1 && g.Games[i + 1] == 1
            count_t += 1
        end
    end

    b2b_per_team[t] = count_t
    total_b2b += count_t
end

println("Total back-to-backs: ", total_b2b)
println("Per team:")
println(b2b_per_team)


Total back-to-backs: 697
Per team:
Dict(5 => 25, 16 => 24, 20 => 25, 12 => 24, 24 => 25, 28 => 21, 8 => 24, 17 => 24, 30 => 22, 1 => 16, 19 => 21, 22 => 22, 23 => 24, 6 => 24, 11 => 24, 9 => 22, 14 => 25, 3 => 24, 29 => 21, 7 => 25, 25 => 22, 4 => 22, 13 => 25, 15 => 20, 2 => 25, 10 => 25, 18 => 24, 21 => 24, 26 => 25, 27 => 23)


In [22]:
using CSV, DataFrames

sched = CSV.read("schedule_final_5000.csv", DataFrame)

Teams = sort(unique(vcat(sched.HomeTeam, sched.AwayTeam)))
min_day = minimum(sched.Day)
max_day = maximum(sched.Day)

function location_of_team_from_sched(t::Int, d::Int, sched::DataFrame)
    day_rows = sched[sched.Day .== d, :]

    idx_away = findfirst(day_rows.AwayTeam .== t)
    if idx_away !== nothing
        return day_rows.HomeTeam[idx_away]
    end

    idx_home = findfirst(day_rows.HomeTeam .== t)
    if idx_home !== nothing
        return t
    end

    return nothing
end

function compute_b2b_stats_from_schedule(sched::DataFrame, Teams, dist)
    b2b_stats = Dict{Int,Dict}()

    max_day = maximum(sched.Day)

    for t in Teams
        total_b2b = 0
        total_miles = 0.0
        distances = Float64[]

        for d in 1:(max_day - 1)
            g1 = sum((sched.Day .== d) .& ((sched.HomeTeam .== t) .| (sched.AwayTeam .== t)))
            g2 = sum((sched.Day .== d + 1) .& ((sched.HomeTeam .== t) .| (sched.AwayTeam .== t)))

            if g1 > 1 || g2 > 1
                @warn "Team $t has $g1 games on day $d or $g2 games on day $(d+1) in schedule; schedule may be invalid."
            end

            if g1 > 0 && g2 > 0
                loc1 = location_of_team_from_sched(t, d, sched)
                loc2 = location_of_team_from_sched(t, d + 1, sched)

                if loc1 !== nothing && loc2 !== nothing
                    d_miles = dist[loc1, loc2]
                    push!(distances, d_miles)
                    total_miles += d_miles
                    total_b2b   += 1
                end
            end
        end

        avg_miles = total_b2b > 0 ? total_miles / total_b2b : 0.0

        b2b_stats[t] = Dict(
            "B2B Count"         => total_b2b,
            "Total Miles"       => total_miles,
            "Avg Miles per B2B" => avg_miles,
            "All Distances"     => distances
        )
    end

    return b2b_stats
end

b2b_stats = compute_b2b_stats_from_schedule(sched, Teams, dist)

println("\n==========================")
println(" B2B AVERAGE DISTANCE DATA")
println("==========================\n")

for t in Teams
    stats = b2b_stats[t]
    println("Team $t:")
    println("   B2B Count:          ", stats["B2B Count"])
    println("   Total B2B Miles:    ", round(stats["Total Miles"], digits=1))
    println("   Avg Miles per B2B:  ", round(stats["Avg Miles per B2B"], digits=1))
    println("   Distances:          ", stats["All Distances"])
    println()
end



 B2B AVERAGE DISTANCE DATA

Team 1:
   B2B Count:          16
   Total B2B Miles:    9212.0
   Avg Miles per B2B:  575.8
   Distances:          [936.0, 1088.0, 805.0, 755.0, 669.0, 16.0, 1.0, 755.0, 1.0, 2169.0, 1968.0, 1.0, 12.0, 12.0, 1.0, 23.0]

Team 2:
   B2B Count:          25
   Total B2B Miles:    24902.0
   Avg Miles per B2B:  996.1
   Distances:          [936.0, 1203.0, 936.0, 936.0, 936.0, 337.0, 484.0, 1121.0, 1258.0, 936.0, 81.0, 1765.0, 936.0, 1156.0, 2143.0, 1493.0, 1940.0, 999.0, 0.0, 407.0, 394.0, 1116.0, 1604.0, 849.0, 936.0]

Team 3:
   B2B Count:          24
   Total B2B Miles:    25758.0
   Avg Miles per B2B:  1073.2
   Distances:          [189.0, 806.0, 189.0, 2285.0, 2693.0, 936.0, 189.0, 1343.0, 1724.0, 484.0, 1121.0, 2591.0, 2591.0, 955.0, 1327.0, 189.0, 1006.0, 550.0, 394.0, 721.0, 189.0, 1493.0, 189.0, 1604.0]

Team 4:
   B2B Count:          22
   Total B2B Miles:    12820.0
   Avg Miles per B2B:  582.7
   Distances:          [532.0, 82.0, 630.0, 189.0, 532.0